# EDA — Mix energetico in Europa: nucleare, rinnovabili e fossili

**Obiettivo del notebook**

Esplorare come è cambiato nel tempo il mix di fonti usate per produrre elettricità nei paesi europei,
con un focus particolare sul confronto tra **nucleare**, **rinnovabili** e **fonti fossili**.

## 1. Fonte dei dati

I dati usati in questo notebook provengono dal dataset **Energy Data** di
[**Our World in Data (OWID)**](https://github.com/owid/energy-data), un progetto di ricerca open-access
dell'Università di Oxford.

- OWID non raccoglie i dati in prima persona, ma li **aggrega e armonizza** da fonti istituzionali
  riconosciute a livello internazionale, principalmente:
  - **Ember** — *Yearly Electricity Data*, dati elettrici annuali (la fonte principale per l'elettricità dal 2000 in poi,
    e dal 1990 per i paesi europei)
  - **Energy Institute** — *Statistical Review of World Energy* (fonte storica, usata per gli anni precedenti)
  - Dati di popolazione e PIL da fonti demografiche/economiche standard usate da OWID
- Il dataset è **aggiornato regolarmente** (ultimo aggiornamento verificato: 2026) ed è openly licensed
  (**CC BY** — riutilizzabile citando la fonte)
- È lo stesso dataset usato per generare i grafici pubblicati su https://ourworldindata.org/energy

**Link diretti:**
- Repository GitHub: https://github.com/owid/energy-data
- File CSV completo: `owid-energy-data.csv` (nella root del repository)
- Codebook (descrizione di ogni colonna): `owid-energy-codebook.csv`
- Pagina di presentazione: https://ourworldindata.org/energy

**Citazione consigliata** (come richiesto da OWID per l'uso dei dati):
> Ember (2026); Energy Institute - Statistical Review of World Energy (2025) — with major processing by Our World in Data.

In [21]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / "data" / "owid-energy-data.csv")

print(df.shape)
df.head()

(23377, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


In [22]:
codebook = pd.read_csv(DATA_DIR / "data" / "owid-energy-codebook.csv")

print(codebook.shape)
codebook.head()

(130, 5)


,column,title,description,unit,source
0,country,Country,Geographic location.,NaN,Our World in Data - Regions (2025)
1,year,Year,Year of observation.,NaN,Our World in Data - Regions (2025)
2,iso_code,ISO code,ISO 3166-1 alpha-3 three-letter country codes.,NaN,International Organization for Standardization...
3,population,Population,"Population by country, available from 10,000 B...",people,Population based on various sources (2024) [ht...
4,gdp,Gross domestic product (GDP),Total economic output of a country or region p...,international-$ in 2011 prices ($),Bolt and van Zanden – Maddison Project Databas...


## 2. Struttura del dataset

Il file è in formato **"long"**: una riga per ogni combinazione di *paese* e *anno*.
Prima di tutto controlliamo dimensioni, periodo temporale coperto e quante "entità" (paesi/aggregati)
sono presenti.


In [23]:
print(f"Righe: {df.shape[0]:,}")
print(f"Colonne: {df.shape[1]}")
print(f"Periodo: {df['year'].min()} - {df['year'].max()}")
print(f"Entità (paesi + aggregati): {df['country'].nunique()}")


Righe: 23,377
Colonne: 130
Periodo: 1900 - 2025
Entità (paesi + aggregati): 314


**Nota importante:** la colonna `country` non contiene *solo* paesi. OWID include anche
**aggregati** (continenti, "World", gruppi di reddito, organizzazioni come "EU (Ember)", "G7 (Ember)" ecc.)
per permettere confronti a livello macro nei propri grafici.

Questi aggregati **non hanno un `iso_code`** (il codice ISO a 3 lettere dei paesi), quindi possiamo
usare questo campo per individuarli ed escluderli quando vogliamo lavorare solo su paesi reali.


In [24]:
# Quante "entità" non sono paesi veri (non hanno iso_code)?
aggregati = sorted(df.loc[df["iso_code"].isna(), "country"].unique())
print(f"Aggregati/non-paesi individuati: {len(aggregati)}")
aggregati[:15]  # primi esempi


Aggregati/non-paesi individuati: 94


['ASEAN (Ember)',
 'Africa',
 'Africa (EI)',
 'Africa (EIA)',
 'Africa (Ember)',
 'Africa (Shift)',
 'Asia',
 'Asia (Ember)',
 'Asia Pacific (EI)',
 'Asia and Oceania (EIA)',
 'Asia and Oceania (Shift)',
 'Australia and New Zealand (EIA)',
 'CIS (EI)',
 'Central America (EI)',
 'Central and South America (EIA)']

## 3. Analisi dei valori mancanti

Vogliamo capire se i `NaN` nei paesi europei sono **strutturali** (gap sparsi nella serie) oppure semplicemente **left-censored**: ogni paese inizia a registrare dati in un anno diverso, e prima di quel momento i valori sono `NaN` per definizione.

Le due situazioni hanno implicazioni molto diverse:
- **Left-censored**: il paese non aveva ancora dati disponibili → i `NaN` sono tutti concentrati nelle righe iniziali e scompaiono una volta che la serie parte.
- **Gap strutturale**: dati mancanti sparsi nel mezzo della serie → possibile per paesi con storie politiche particolari (es. ex-URSS, ex-Jugoslavia) o per variabili non rilevate sistematicamente.

La strategia: per ogni paese e ogni colonna chiave, verifico se tutti i `NaN` cadono *prima* del primo valore valido (`first_valid_year`). Se sì, il pattern è left-censored; altrimenti c'è almeno un gap interno.

In [25]:
EUROPE_ISO = {
    "ALB", "AND", "AUT", "BLR", "BEL", "BIH", "BGR", "HRV", "CYP", "CZE",
    "DNK", "EST", "FIN", "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ITA",
    "XKX", "LVA", "LIE", "LTU", "LUX", "MLT", "MDA", "MCO", "MNE", "NLD",
    "MKD", "NOR", "POL", "PRT", "ROU", "RUS", "SMR", "SRB", "SVK", "SVN",
    "ESP", "SWE", "CHE", "UKR", "GBR", "VAT",
}

KEY_COLS = [
    "electricity_generation",
    "fossil_electricity",
    "nuclear_electricity",
    "renewables_electricity",
    "solar_electricity",
    "wind_electricity",
    "hydro_electricity",
]

df_eu = df[df["iso_code"].isin(EUROPE_ISO)].copy()
print(f"Paesi europei nel dataset: {df_eu['country'].nunique()}")
print(f"Righe totali: {len(df_eu):,}  |  Anni: {df_eu['year'].min()}–{df_eu['year'].max()}")

Paesi europei nel dataset: 40
Righe totali: 3,488  |  Anni: 1900–2025


In [26]:
def classify_missing(series: pd.Series) -> dict:
    """
    Classifica il pattern dei NaN in una serie temporale ordinata per anno.
    - 'left-censored'  : NaN solo prima del primo valore valido, poi serie continua
    - 'internal-gaps'  : NaN presenti anche dopo che la serie è iniziata
    - 'always-null'    : nessun dato disponibile
    - 'complete'       : nessun NaN
    """
    first_idx = series.first_valid_index()
    if first_idx is None:
        return {"first_valid": None, "nan_before": int(series.isna().sum()), "nan_after": 0, "pattern": "always-null"}

    nan_before = int(series.loc[:first_idx].iloc[:-1].isna().sum())
    nan_after  = int(series.loc[first_idx:].isna().sum())

    if nan_after == 0:
        pattern = "complete" if nan_before == 0 else "left-censored"
    else:
        pattern = "internal-gaps"

    return {"first_valid": first_idx, "nan_before": nan_before, "nan_after": nan_after, "pattern": pattern}


records = []
for country, grp in df_eu.groupby("country"):
    grp_sorted = grp.sort_values("year").set_index("year")
    for col in KEY_COLS:
        if col not in grp_sorted.columns:
            continue
        info = classify_missing(grp_sorted[col])
        records.append({"country": country, "column": col, **info})

miss = pd.DataFrame(records)
miss.head(10)

,country,column,first_valid,nan_before,nan_after,pattern
0,Albania,electricity_generation,2000,100,0,left-censored
1,Albania,fossil_electricity,2000,100,0,left-censored
2,Albania,nuclear_electricity,2000,100,0,left-censored
3,Albania,renewables_electricity,2000,100,0,left-censored
4,Albania,solar_electricity,2000,100,0,left-censored
5,Albania,wind_electricity,2000,100,0,left-censored
6,Albania,hydro_electricity,2000,100,0,left-censored
7,Austria,electricity_generation,1985,85,0,left-censored
8,Austria,fossil_electricity,1990,90,0,left-censored
9,Austria,nuclear_electricity,1965,65,0,left-censored


In [27]:
# Riepilogo: quante coppie paese-colonna per ciascun pattern
print("=== Distribuzione dei pattern (paese × colonna) ===")
print(miss["pattern"].value_counts().to_string())

print("\n=== Paesi/colonne con gap INTERNI (non solo left-censored) ===")
gaps = miss[miss["pattern"] == "internal-gaps"][["country", "column", "first_valid", "nan_after"]]
print(gaps.sort_values("nan_after", ascending=False).to_string(index=False))

=== Distribuzione dei pattern (paese × colonna) ===
pattern
left-censored    174
complete          86
internal-gaps     20

=== Paesi/colonne con gap INTERNI (non solo left-censored) ===
        country                 column  first_valid  nan_after
North Macedonia       wind_electricity         1990         14
       Slovenia       wind_electricity         1990         13
          Malta       wind_electricity         1990         12
         Serbia       wind_electricity         1990         12
        Romania       wind_electricity         1965          5
         Cyprus       wind_electricity         1965          4
        Croatia       wind_electricity         1990          4
       Bulgaria       wind_electricity         1965          4
      Lithuania       wind_electricity         1985          4
       Slovakia       wind_electricity         1965          3
        Ukraine renewables_electricity         1985          2
        Estonia       wind_electricity         1985      

In [28]:
# Anno di inizio per ogni paese × colonna (solo paesi con almeno un dato)
pivot_start = miss[miss["pattern"] != "always-null"].pivot(
    index="country", columns="column", values="first_valid"
).astype("Int64")  # Int64 per mostrare <NA> invece di NaN float

print("=== Anno del primo dato valido (paese × colonna) ===")
print(pivot_start.to_string())

print("\n=== Dettaglio gap INTERNI: anni con NaN dopo l'inizio della serie ===")
for country, grp in df_eu.groupby("country"):
    grp_sorted = grp.sort_values("year").set_index("year")
    for col in KEY_COLS:
        if col not in grp_sorted.columns:
            continue
        s = grp_sorted[col]
        first_idx = s.first_valid_index()
        if first_idx is None:
            continue
        gap_years = s.loc[first_idx:][s.loc[first_idx:].isna()].index.tolist()
        if gap_years:
            ranges = []
            start = gap_years[0]
            prev = gap_years[0]
            for y in gap_years[1:]:
                if y == prev + 1:
                    prev = y
                else:
                    ranges.append(f"{start}–{prev}" if start != prev else str(start))
                    start = prev = y
            ranges.append(f"{start}–{prev}" if start != prev else str(start))
            print(f"  {country:30s}  {col:30s}  gaps: {', '.join(ranges)}")

=== Anno del primo dato valido (paese × colonna) ===
column                  electricity_generation  fossil_electricity  hydro_electricity  nuclear_electricity  renewables_electricity  solar_electricity  wind_electricity
country                                                                                                                                                                
Albania                                   2000                2000               2000                 2000                    2000               2000              2000
Austria                                   1985                1990               1965                 1965                    1965               1965              1965
Belarus                                   1985                2000               1985                 1985                    1985               1985              1985
Belgium                                   1985                1990               1965                 1965 